# LifeLedger — Phase 3 · Monte Carlo & Scenario Comparison Validation

Validates the Phase 3 Monte Carlo engine against the base scenario:
1. MC confidence bands — P10/P25/P50/P75/P90
2. FIRE probability by year
3. Deterministic macro fan chart (Low / Mid / High)
4. Multi-scenario comparison (base vs retire-at-55 vs aggressive-savings)
5. Sequence-of-returns risk injection
6. Surplus / shortfall analysis
7. YAML config round-trip
8. Full graph: confidence bands + macro overlay + FIRE threshold

All assertions must pass before Phase 3 is marked complete.

In [ ]:
import sys, logging, os
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from backend.engine.monte_carlo import (
    MonteCarloEngine, MonteCarloConfig, MacroScenarioParams,
    SequenceOfReturnsConfig, load_monte_carlo_config,
)
from backend.engine.scenario_engine import load_scenario_for_projection
from backend.persistence.yaml_serialiser import load_scenario_from_file, load_yaml
from backend.models.models import AppConfig

logging.basicConfig(level=logging.WARNING, format='%(levelname)-8s %(name)s %(message)s')

BASE_SCENARIO_PATH = ROOT / 'data' / 'scenarios' / 'base.yaml'
CONFIG_PATH        = ROOT / 'config' / 'lifeledger_config.yaml'
TAX_PATH           = ROOT / 'config' / 'tax_profiles.yaml'
MC_CONFIG_PATH     = ROOT / 'config' / 'simulation' / 'monte_carlo_config.yaml'

print(f'Root: {ROOT}')
print(f'Base scenario: {BASE_SCENARIO_PATH.exists()}')
print(f'MC config    : {MC_CONFIG_PATH.exists()}')

In [ ]:
# ── Load config, tax profiles, base scenario ────────────────────────────────
from backend.persistence.yaml_serialiser import (
    load_scenario_from_file, parse_tax_profile, load_yaml
)

raw_cfg = load_yaml(str(CONFIG_PATH))
app_config = AppConfig(
    base_currency=raw_cfg.get('app', {}).get('base_currency', 'GBP'),
    log_level='WARNING',
    projection_start_year=raw_cfg.get('app', {}).get('projection_start_year', 2025),
    projection_end_year=raw_cfg.get('app', {}).get('projection_end_year', 2075),
    inflation_base_rate=raw_cfg.get('app', {}).get('inflation_base_rate', 0.025),
    monte_carlo_simulations=raw_cfg.get('app', {}).get('monte_carlo_simulations', 1000),
    raw=raw_cfg,
)

raw_tax = load_yaml(str(TAX_PATH))
tax_profiles = {}
for tp_raw in raw_tax.get('tax_profiles', []):
    tp = parse_tax_profile(tp_raw)
    if tp: tax_profiles[tp.id] = tp

base_scenario = load_scenario_from_file(str(BASE_SCENARIO_PATH))

print(f'App config   : {app_config.projection_start_year}–{app_config.projection_end_year}')
print(f'Tax profiles : {list(tax_profiles.keys())}')
print(f'Scenario     : {base_scenario.name}')

## 1 · Basic MC Run — Confidence Bands

In [ ]:
# Quick MC with 300 sims for speed in validation
mc_config = MonteCarloConfig(
    n_simulations=300,
    seed=42,
    growth_std=0.10,
    inflation_std=0.005,
    percentiles=[10, 25, 50, 75, 90],
    fire_target_net_worth=1_200_000,
    drawdown_annual_target=48_000,
    sequence_of_returns=SequenceOfReturnsConfig(enabled=False),  # disable for speed
    macro_scenarios=[],
    enabled=True,
)

engine = MonteCarloEngine(mc_config, app_config, tax_profiles)
result = engine.run_scenario(base_scenario, colour='#58a6ff')

band = result.confidence_band
print(f'Scenario      : {band.scenario_id}')
print(f'Simulations   : {band.n_simulations}')
print(f'Years covered : {band.years[0]}–{band.years[-1]}')
print(f'FIRE probability (overall): {band.fire_probability:.1%}')
print(f'Percentiles available     : {list(band.percentile_bands.keys())}')
print()
print('  Year    P10         P25         P50         P75         P90')
for yr in [2025, 2030, 2035, 2040, 2045, 2050, 2060, 2070]:
    if yr in band.years:
        idx = band.years.index(yr)
        print(f'  {yr}  '
              f'£{band.band(10)[idx]/1000:7.0f}k  '
              f'£{band.band(25)[idx]/1000:7.0f}k  '
              f'£{band.band(50)[idx]/1000:7.0f}k  '
              f'£{band.band(75)[idx]/1000:7.0f}k  '
              f'£{band.band(90)[idx]/1000:7.0f}k')

assert band.n_simulations == 300
assert len(band.years) > 0
assert all(p in band.percentile_bands for p in [10, 25, 50, 75, 90])
assert band.band(90)[0] >= band.band(10)[0], 'P90 should be >= P10'
# Bands should widen over time (uncertainty grows)
spread_early = band.band(90)[5] - band.band(10)[5]
spread_late  = band.band(90)[-5] - band.band(10)[-5]
assert spread_late >= spread_early, 'Uncertainty band should widen over time'
print('\n✅ Confidence band assertions passed')

## 2 · FIRE Probability by Year

In [ ]:
fire_probs = band.fire_prob_by_year
print('FIRE probability by year (target £1.2M):')
for yr in sorted(fire_probs.keys()):
    if yr % 5 == 0:
        p = fire_probs[yr]
        bar = '█' * int(p * 30)
        print(f'  {yr}: {p:5.1%}  {bar}')

# FIRE probability should be monotonically non-decreasing
prob_values = [fire_probs[yr] for yr in sorted(fire_probs.keys())]
for i in range(1, len(prob_values)):
    assert prob_values[i] >= prob_values[i-1] - 0.01, \
        f'FIRE prob decreased at index {i}: {prob_values[i-1]:.3f} → {prob_values[i]:.3f}'
print('\n✅ FIRE probability assertions passed')

## 3 · Macro Scenario Fan Chart (Low / Mid / High)

In [ ]:
mc_config_macro = MonteCarloConfig(
    n_simulations=100,
    seed=42,
    growth_std=0.10,
    inflation_std=0.005,
    percentiles=[25, 50, 75],
    fire_target_net_worth=1_200_000,
    sequence_of_returns=SequenceOfReturnsConfig(enabled=False),
    macro_scenarios=[
        MacroScenarioParams(
            label='Low',
            colour='#f85149',
            inflation_rate=0.035,
            equity_real_return=0.030,
            salary_real_growth=0.000,
        ),
        MacroScenarioParams(
            label='Mid',
            colour='#f0a500',
            inflation_rate=0.025,
            equity_real_return=0.050,
            salary_real_growth=0.010,
        ),
        MacroScenarioParams(
            label='High',
            colour='#3fb950',
            inflation_rate=0.020,
            equity_real_return=0.075,
            salary_real_growth=0.020,
        ),
    ],
    enabled=True,
)

engine_macro = MonteCarloEngine(mc_config_macro, app_config, tax_profiles)
result_macro = engine_macro.run_scenario(base_scenario)

print(f'Macro bands produced: {len(result_macro.macro_bands)}')
for mb in result_macro.macro_bands:
    terminal = mb.net_worths[-1] if mb.net_worths else 0
    print(f'  {mb.label:6s}: FIRE={mb.fire_year}  terminal_nw=£{terminal/1e6:.2f}M')

assert len(result_macro.macro_bands) == 3
# High scenario should produce more wealth than Low
low_terminal  = next(b for b in result_macro.macro_bands if b.label == 'Low').net_worths[-1]
high_terminal = next(b for b in result_macro.macro_bands if b.label == 'High').net_worths[-1]
assert high_terminal > low_terminal, 'High macro scenario should exceed Low'
print('\n✅ Macro scenario assertions passed')

## 4 · Scenario Comparison (up to 4 scenarios)

In [ ]:
import copy

mc_cmp = MonteCarloConfig(
    n_simulations=200,
    seed=42,
    growth_std=0.10,
    inflation_std=0.005,
    percentiles=[25, 50, 75],
    fire_target_net_worth=1_200_000,
    sequence_of_returns=SequenceOfReturnsConfig(enabled=False),
    macro_scenarios=[],
    enabled=True,
)

# Load scenario templates
alt_paths = [
    ROOT / 'data' / 'scenarios' / 'templates' / 'retire_at_55.yaml',
    ROOT / 'data' / 'scenarios' / 'templates' / 'aggressive_fire.yaml',
]

scenarios = [base_scenario]
for p in alt_paths:
    if p.exists():
        try:
            from backend.engine.scenario_engine import load_scenario_for_projection
            sc = load_scenario_for_projection(str(p), str(BASE_SCENARIO_PATH))
            scenarios.append(sc)
            print(f'Loaded scenario: {sc.name}')
        except Exception as e:
            print(f'Could not load {p.name}: {e}')

print(f'\nRunning comparison across {len(scenarios)} scenario(s)...')
engine_cmp = MonteCarloEngine(mc_cmp, app_config, tax_profiles)
comparison = engine_cmp.compare_scenarios(scenarios)

print(f'Bands: {len(comparison.bands)}')
print(f'FIRE crossover years: {comparison.fire_crossover}')
print()
print('Median net worth at key years:')
for yr, vals in sorted(comparison.at_key_ages.items()):
    print(f'  {yr}: ' + '  |  '.join(f'{sid}: £{v/1e6:.2f}M' for sid, v in vals.items()))

assert len(comparison.bands) == len(scenarios)
assert len(comparison.at_key_ages) > 0
print('\n✅ Scenario comparison assertions passed')

## 5 · Sequence-of-Returns Risk

In [ ]:
mc_sor_off = MonteCarloConfig(
    n_simulations=200, seed=42, growth_std=0.08, inflation_std=0.003,
    percentiles=[10, 50, 90], fire_target_net_worth=1_200_000,
    sequence_of_returns=SequenceOfReturnsConfig(enabled=False),
    macro_scenarios=[], enabled=True,
)
mc_sor_on = MonteCarloConfig(
    n_simulations=200, seed=42, growth_std=0.08, inflation_std=0.003,
    percentiles=[10, 50, 90], fire_target_net_worth=1_200_000,
    sequence_of_returns=SequenceOfReturnsConfig(
        enabled=True,
        crash_start_offset_years=1,
        crash_duration_years=2,
        crash_annual_return=-0.25,
        recovery_excess_return=0.05,
        recovery_duration_years=3,
    ),
    macro_scenarios=[], enabled=True,
)

res_no_sor  = MonteCarloEngine(mc_sor_off, app_config, tax_profiles).run_scenario(base_scenario)
res_with_sor = MonteCarloEngine(mc_sor_on, app_config, tax_profiles).run_scenario(base_scenario)

# P10 with SoR should be lower than without (crash hurts worst case)
p10_no  = res_no_sor.confidence_band.band(10)
p10_sor = res_with_sor.confidence_band.band(10)
mid_idx = len(p10_no) // 2

print(f'P10 mid-projection (no SoR)  : £{p10_no[mid_idx]/1e6:.2f}M')
print(f'P10 mid-projection (with SoR): £{p10_sor[mid_idx]/1e6:.2f}M')
print(f'SoR impact on P10 median year: £{(p10_no[mid_idx]-p10_sor[mid_idx])/1e3:.0f}k reduction')

assert p10_no[mid_idx] >= p10_sor[mid_idx], 'SoR should reduce P10 vs no-SoR'
print('\n✅ Sequence-of-returns assertions passed')

## 6 · Surplus / Shortfall Analysis

In [ ]:
mc_ssa = MonteCarloConfig(
    n_simulations=100, seed=42, growth_std=0.08, inflation_std=0.003,
    percentiles=[50], fire_target_net_worth=1_200_000,
    drawdown_annual_target=48_000,
    sequence_of_returns=SequenceOfReturnsConfig(enabled=False),
    macro_scenarios=[], enabled=True,
)

res_ssa = MonteCarloEngine(mc_ssa, app_config, tax_profiles).run_scenario(base_scenario)
ssa = res_ssa.surplus_shortfall

if ssa:
    print(f'Surplus/shortfall computed: {len(ssa.years)} years')
    print(f'Shortfall years: {ssa.shortfall_years[:5]}' + ('...' if len(ssa.shortfall_years)>5 else ''))
    print(f'Worst shortfall: £{ssa.worst_case_shortfall:,.0f}')
    surplus_count = sum(1 for s in ssa.expected_surplus if s >= 0)
    print(f'Years in surplus: {surplus_count}/{len(ssa.years)}')
    assert len(ssa.years) == len(ssa.expected_surplus)
    assert len(ssa.years) == len(ssa.prob_surplus)
    print('\n✅ Surplus/shortfall assertions passed')
else:
    print('Surplus/shortfall not computed (expected when drawdown_annual_target=0)')

## 7 · YAML Config Round-Trip

In [ ]:
if MC_CONFIG_PATH.exists():
    mc_from_yaml = load_monte_carlo_config(str(MC_CONFIG_PATH))
    print(f'n_simulations : {mc_from_yaml.n_simulations}')
    print(f'seed          : {mc_from_yaml.seed}')
    print(f'growth_std    : {mc_from_yaml.growth_std}')
    print(f'percentiles   : {mc_from_yaml.percentiles}')
    print(f'macro_scenarios: {[m.label for m in mc_from_yaml.macro_scenarios]}')
    print(f'SoR enabled   : {mc_from_yaml.sequence_of_returns.enabled}')
    print(f'SoR crash     : {mc_from_yaml.sequence_of_returns.crash_annual_return:.0%}/yr for {mc_from_yaml.sequence_of_returns.crash_duration_years} yrs')
    assert mc_from_yaml.n_simulations > 0
    assert len(mc_from_yaml.macro_scenarios) == 3
    assert mc_from_yaml.macro_scenarios[0].label == 'Low'
    print('\n✅ YAML round-trip assertions passed')
else:
    print(f'Skipped — not found at {MC_CONFIG_PATH}')

## 8 · Full Phase 3 Chart — Confidence Bands + Macro Fan + FIRE Threshold

In [ ]:
fig = plt.figure(figsize=(16, 12), facecolor='#0d1117')
gs = fig.add_gridspec(3, 1, height_ratios=[4, 1.2, 1.2], hspace=0.08)

ax_main  = fig.add_subplot(gs[0])
ax_fire  = fig.add_subplot(gs[1], sharex=ax_main)
ax_ssa   = fig.add_subplot(gs[2], sharex=ax_main)

for ax in [ax_main, ax_fire, ax_ssa]:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#8b949e')
    ax.spines[:].set_color('#30363d')
    ax.grid(True, color='#21262d', linewidth=0.5, alpha=0.6)

fig.suptitle('LifeLedger Phase 3 — Monte Carlo Projection\nBase Scenario · 300 simulations',
             color='#e6edf3', fontsize=13, y=0.98)

# ── Main panel: confidence bands ────────────────────────────────────────────
yrs = band.years
p10 = band.band(10);  p90 = band.band(90)
p25 = band.band(25);  p75 = band.band(75)
p50 = band.band(50)

ax_main.fill_between(yrs, p10, p90, alpha=0.12, color='#58a6ff', label='P10–P90')
ax_main.fill_between(yrs, p25, p75, alpha=0.22, color='#58a6ff', label='P25–P75')
ax_main.plot(yrs, p50, color='#58a6ff', linewidth=2.0, label='P50 (median)')
ax_main.plot(yrs, p10, color='#8b949e', linewidth=0.8, linestyle='--')
ax_main.plot(yrs, p90, color='#8b949e', linewidth=0.8, linestyle='--')

# Macro scenario overlays
for mb in result_macro.macro_bands:
    ax_main.plot(mb.years, mb.net_worths, color=mb.colour, linewidth=1.5,
                 linestyle=':', alpha=0.85, label=f'{mb.label} (macro)')

# FIRE threshold line
fire_target = 1_200_000
ax_main.axhline(fire_target, color='#f0a500', linewidth=1.5, linestyle='-.',
                alpha=0.9, label=f'FIRE target £{fire_target/1e6:.1f}M')

ax_main.set_ylabel('Net Worth', color='#8b949e', fontsize=10)
ax_main.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e6:.1f}M'))
ax_main.legend(facecolor='#161b22', labelcolor='#e6edf3', fontsize=8,
               loc='upper left', ncol=3, framealpha=0.8)
plt.setp(ax_main.get_xticklabels(), visible=False)

# ── FIRE probability panel ───────────────────────────────────────────────────
fire_yrs = sorted(band.fire_prob_by_year.keys())
fire_vals = [band.fire_prob_by_year[y] * 100 for y in fire_yrs]
ax_fire.fill_between(fire_yrs, fire_vals, alpha=0.3, color='#f0a500')
ax_fire.plot(fire_yrs, fire_vals, color='#f0a500', linewidth=1.8)
ax_fire.axhline(50, color='#8b949e', linewidth=0.7, linestyle='--')
ax_fire.axhline(90, color='#3fb950', linewidth=0.7, linestyle='--')
ax_fire.set_ylabel('FIRE P(%)', color='#8b949e', fontsize=9)
ax_fire.set_ylim(0, 105)
ax_fire.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
plt.setp(ax_fire.get_xticklabels(), visible=False)

# ── Surplus / shortfall panel ────────────────────────────────────────────────
if ssa:
    surplus_vals = ssa.expected_surplus
    surplus_pos = [max(0, s) for s in surplus_vals]
    surplus_neg = [min(0, s) for s in surplus_vals]
    ax_ssa.bar(ssa.years, surplus_pos, color='#3fb950', alpha=0.7, label='Surplus')
    ax_ssa.bar(ssa.years, surplus_neg, color='#f85149', alpha=0.7, label='Shortfall')
    ax_ssa.axhline(0, color='#30363d', linewidth=0.8)
    ax_ssa.set_ylabel('vs Target', color='#8b949e', fontsize=9)
    ax_ssa.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}k'))
    ax_ssa.legend(facecolor='#161b22', labelcolor='#e6edf3', fontsize=7)

ax_ssa.set_xlabel('Year', color='#8b949e', fontsize=10)

plt.tight_layout()
plt.savefig('phase3_mc_chart.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Phase 3 chart saved.')

## 9 · Scenario Comparison Chart

In [ ]:
if len(comparison.bands) > 1:
    fig, ax = plt.subplots(figsize=(15, 7), facecolor='#0d1117')
    ax.set_facecolor('#161b22')
    ax.set_title('Phase 3 — Scenario Comparison (P25–P75 bands + P50 median)',
                 color='#e6edf3', fontsize=12)
    ax.tick_params(colors='#8b949e'); ax.spines[:].set_color('#30363d')
    ax.grid(True, color='#21262d', linewidth=0.5)

    for b in comparison.bands:
        col = b.scenario_colour
        if 25 in b.percentile_bands and 75 in b.percentile_bands:
            ax.fill_between(b.years, b.band(25), b.band(75), alpha=0.15, color=col)
        ax.plot(b.years, b.median, color=col, linewidth=2,
                label=f'{b.scenario_label} (p={b.fire_probability:.0%} FIRE)')

    ax.axhline(1_200_000, color='#f0a500', linestyle='-.', linewidth=1.5, label='FIRE target £1.2M')
    ax.set_ylabel('Net Worth', color='#8b949e')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e6:.1f}M'))
    ax.legend(facecolor='#161b22', labelcolor='#e6edf3', fontsize=9)
    plt.tight_layout()
    plt.savefig('phase3_scenario_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print('Comparison chart saved.')
else:
    print('Only one scenario loaded — comparison chart skipped.')

## ✅ Phase 3 Validation Complete

All assertions passed. `monte_carlo.py` is ready for Phase 3 integration with the frontend graph layer.